In [2]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = "1"
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib as plt
from PIL import Image
sys.path.append('../src')
from utils import *
from matplotlib.lines import Line2D
import matplotlib as mpl
import math
import matplotlib.image as mpimg
import scipy.stats as stats
import scipy as sp
from scipy.stats import pearsonr,binom,linregress
from ast import literal_eval
import json
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.stats import gaussian_kde
os.chdir("..")

In [13]:
for mode in ["vision", "imagery"]:
    file_paths = []
    for i, method in enumerate(["mindeye", 'mindeye_mcd_5passes_0.15dr', "mindeye_mcd_10passes_0.15dr", "mindeye_mcd_25passes_0.15dr"]):
        
        df = pd.DataFrame()
        for subject in [1]:
            new_df = pd.read_csv(f"output/mental_imagery_paper_b3/{mode}/{method}/subject{subject}_statistics_18.csv")
            df = pd.concat([df, new_df])

        df.to_csv(f"output/mental_imagery_paper_b3/{mode}/{method}/all_subject_statistics.csv")

        df.reset_index(drop=True, inplace=True)
        # Define the columns to be used for selecting the best row for each "ID"
        columns_to_consider = [
            "CLIP Cosine",
            # "Brain Correlation V1",
            # "Brain Correlation V2",
            # "Brain Correlation V3",
            # "Brain Correlation V4",
            # "Brain Correlation Higher Visual",
            "Brain Correlation NSD General",
            "AlexNet 2",
            "AlexNet 3",
            "AlexNet 4",
            "AlexNet 5",
            "AlexNet 6",
            "AlexNet 7",
            "CLIP Two-way",
            "Inception V3"
        ]
        filtered_df = df[(df['Sample Indicator'] == 12)]
        # Rank each of the relevant columns across the filtered DataFrame and sum these ranks for each row
        ranked_df = filtered_df[columns_to_consider].rank(ascending=False, method='min', na_option='keep')
        filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)

        # Find the row with the lowest sum of ranks for each "ID"
        best_rows_indices_filtered = filtered_df.groupby('ID')['Rank_Sum'].idxmin()
        best_rows_filtered = filtered_df.loc[best_rows_indices_filtered].sort_values(by="ID")

        # Generate file paths for the images corresponding to the rows after filtering
        image_file_paths = [
            f"output/mental_imagery_paper_b3/{row['Mode']}/{row['Method']}/subject{int(row['Subject'])}/{int(row['ID'])}/{int(row['Sample Count'])}.png"
            for index, row in best_rows_filtered.iterrows()
        ]

        # Generate file paths for the ground truth images corresponding to the filtered rows
        ground_truth_file_paths = [
            f"output/mental_imagery_paper_b3/{row['Mode']}/{row['Method']}/subject{int(row['Subject'])}/{int(row['ID'])}/ground_truth.png"
            for index, row in best_rows_filtered.iterrows()
        ]
        # print(len(image_file_paths))
        if i == 0:
            file_paths += ground_truth_file_paths
        file_paths += image_file_paths

    captions = ["Ground Truth", "MindEye1", "ME1 + BOI", "ME1 + COI-Last", "ME1 + COI-Hidden"]
    images = []
    for file in file_paths:
        images.append(Image.open(file))
    if mode == "vision":
        title =  f"Reconstructions for seen stimuli"
    else:
        title =  f"Reconstructions for imagined stimuli"
        
        
    tiled_figure = format_tiled_figure(
        images=images,
        captions=captions, 
        rows = 5, 
        cols = 18,
        red_line_index=0,  # Replace with the index of the column/row to highlight red
        buffer=20, # The buffer space in pixels between images.
        mode=3, # The mode of how captions and titles are used.
        title = title
        )
    tiled_figure.save(f"output/mental_imagery_paper_b3/image_grids/mi_coi_{mode}_v3.png")

/tmp/ipykernel_3779/38701881.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)
/tmp/ipykernel_3779/38701881.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)
/tmp/ipykernel_3779/38701881.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pan

In [4]:
for mode in ["vision", "imagery"]:
    file_paths = []
    for i, method in enumerate(["mindeye_subj01_hypatia_default", 'mindeye_subj01_hypatia_0.4_dropout_mlp', "mindeye_subj01_hypatia_0.6_dropout_mlp", "mindeye_subj01_hypatia_0.8_dropout_mlp", "mindeye_subj01_hypatia_0.9_dropout_mlp"]):
        
        df = pd.DataFrame()
        for subject in [1]:
            new_df = pd.read_csv(f"output/mental_imagery_paper_b3/{mode}/{method}/subject{subject}_statistics_18.csv")
            df = pd.concat([df, new_df])

        # df.to_csv(f"output/mental_imagery_paper_b3/{mode}/{method}/all_subject_statistics.csv")

        df.reset_index(drop=True, inplace=True)
        # Define the columns to be used for selecting the best row for each "ID"
        columns_to_consider = [
            "CLIP Cosine",
            # "Brain Correlation V1",
            # "Brain Correlation V2",
            # "Brain Correlation V3",
            # "Brain Correlation V4",
            # "Brain Correlation Higher Visual",
            "Brain Correlation NSD General",
            "AlexNet 2",
            "AlexNet 3",
            "AlexNet 4",
            "AlexNet 5",
            "AlexNet 6",
            "AlexNet 7",
            "CLIP Two-way",
            "Inception V3"
        ]
        filtered_df = df[(df['Sample Indicator'] == 12)]
        # Rank each of the relevant columns across the filtered DataFrame and sum these ranks for each row
        ranked_df = filtered_df[columns_to_consider].rank(ascending=False, method='min', na_option='keep')
        filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)

        # Find the row with the lowest sum of ranks for each "ID"
        best_rows_indices_filtered = filtered_df.groupby('ID')['Rank_Sum'].idxmin()
        best_rows_filtered = filtered_df.loc[best_rows_indices_filtered].sort_values(by="ID")

        # Generate file paths for the images corresponding to the rows after filtering
        image_file_paths = [
            f"output/mental_imagery_paper_b3/{row['Mode']}/{row['Method']}/subject{int(row['Subject'])}/{int(row['ID'])}/{int(row['Sample Count'])}.png"
            for index, row in best_rows_filtered.iterrows()
        ]

        # Generate file paths for the ground truth images corresponding to the filtered rows
        ground_truth_file_paths = [
            f"output/mental_imagery_paper_b3/{row['Mode']}/{row['Method']}/subject{int(row['Subject'])}/{int(row['ID'])}/ground_truth.png"
            for index, row in best_rows_filtered.iterrows()
        ]
        # print(len(image_file_paths))
        if i == 0:
            file_paths += ground_truth_file_paths
        file_paths += image_file_paths

    captions = ["Ground Truth", "MindEye1", "ME1 Hypatia", "ME1 0.4 Dropout"]
    images = []
    for file in file_paths:
        images.append(Image.open(file))
    if mode == "vision":
        title =  f"Reconstructions for seen stimuli"
    else:
        title =  f"Reconstructions for imagined stimuli"
        
        
    tiled_figure = format_tiled_figure(
        images=images,
        captions=captions, 
        rows = 4, 
        cols = 18,
        red_line_index=0,  # Replace with the index of the column/row to highlight red
        buffer=20, # The buffer space in pixels between images.
        mode=3, # The mode of how captions and titles are used.
        title = title
        )
    tiled_figure.save(f"output/mental_imagery_paper_b3/image_grids/mi_mindeye_dropout_{mode}.png")

/tmp/ipykernel_116962/1941262745.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)
/tmp/ipykernel_116962/1941262745.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)
/tmp/ipykernel_116962/1941262745.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.py

In [18]:
for mode in ["vision", "imagery"]:
    file_paths = []
    captions = []  
    method = "mindeye_mcd_25passes_0.4dr"
    # for i, method in enumerate(["mindeye_mcd_25passes_0.15dr", "mindeye_mcd_25passes_0.4dr"]):
    for i, rep in enumerate(range(10)):
        
        df = pd.read_csv(f"output/mental_imagery_paper_b3/{mode}/{method}/subject1_statistics_18.csv")
        
        # Define the columns to be used for selecting the best row for each "ID"
        columns_to_consider = [
            "CLIP Cosine",
            # "Brain Correlation V1",
            # "Brain Correlation V2",
            # "Brain Correlation V3",
            # "Brain Correlation V4",
            # "Brain Correlation Higher Visual",
            "Brain Correlation NSD General",
            "AlexNet 2",
            "AlexNet 3",
            "AlexNet 4",
            "AlexNet 5",
            "AlexNet 6",
            "AlexNet 7",
            "CLIP Two-way",
            "Inception V3"
        ]
        filtered_df = df[(df['Sample Indicator'] == 12) & (df['Sample Count'] == rep)]
        # Rank each of the relevant columns across the filtered DataFrame and sum these ranks for each row
        ranked_df = filtered_df[columns_to_consider].rank(ascending=False, method='min', na_option='keep')
        filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)

        # Find the row with the lowest sum of ranks for each "ID"
        best_rows_indices_filtered = filtered_df.groupby('ID')['Rank_Sum'].idxmin()
        best_rows_filtered = filtered_df.loc[best_rows_indices_filtered].sort_values(by="ID")

        # Generate file paths for the images corresponding to the rows after filtering
        image_file_paths = [
            f"output/mental_imagery_paper_b3/{row['Mode']}/{row['Method']}/subject{int(row['Subject'])}/{int(row['ID'])}/{int(row['Sample Count'])}.png"
            for index, row in best_rows_filtered.iterrows()
        ]

        # Generate file paths for the ground truth images corresponding to the filtered rows
        ground_truth_file_paths = [
            f"output/mental_imagery_paper_b3/{row['Mode']}/{row['Method']}/subject{int(row['Subject'])}/{int(row['ID'])}/ground_truth.png"
            for index, row in best_rows_filtered.iterrows()
        ]
        # print(len(image_file_paths))
        if i == 0:
            file_paths += ground_truth_file_paths
            captions += [f"Ground Truth" for index, row in best_rows_filtered.iterrows()]
        file_paths += image_file_paths
        captions += [f"MCD Var: {row['MCD Var']:.4f}" for index, row in best_rows_filtered.iterrows()]

    # captions = ["Ground Truth", "ME1 0.15 Dropout", "ME1 0.4 Dropout"]
    images = []
    for file in file_paths:
        images.append(Image.open(file))
    if mode == "vision":
        title =  f"Reconstructions for seen stimuli"
    else:
        title =  f"Reconstructions for imagined stimuli"
        
        
    # tiled_figure = format_tiled_figure(
    #     images=images,
    #     captions=captions, 
    #     rows = 2, 
    #     cols = 18,
    #     # red_line_index=0,  # Replace with the index of the column/row to highlight red
    #     buffer=100, # The buffer space in pixels between images.
    #     mode=0, # The mode of how captions and titles are used.
    #     title = title
    #     )
    tiled_figure = tileImages(title, images, captions, 11, 18)
    tiled_figure.save(f"output/mental_imagery_paper_b3/image_grids/mi_{method}_MCD_Var_single_trial_{mode}.png")

/tmp/ipykernel_93300/4124931536.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)
/tmp/ipykernel_93300/4124931536.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Rank_Sum'] = ranked_df.sum(axis=1)
/tmp/ipykernel_93300/4124931536.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat